In [8]:
# %pip install torch 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [43]:
import pandas as pd
df=pd.read_csv("../data/cleaned_reviews.csv")

print("Shape of the dataset:", df.shape)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["Text"]
y = df["Sentiment"]
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print(y_train.value_counts())

from sklearn.utils import resample
import pandas as pd

train_df = pd.DataFrame({
    "Text": X_train,
    "Sentiment": y_train
})

positive = train_df[train_df["Sentiment"] == "Positive"]
other = train_df[train_df["Sentiment"] != "Positive"]

positive_sampled = resample(
    positive,
    n_samples=60000,
    random_state=42,
    replace=False
)

train_df = pd.concat([other, positive_sampled])
train_df = train_df.sample(frac=1, random_state=42)

X_train = train_df["Text"]
y_train = train_df["Sentiment"]

print(y_train.value_counts())
print("New training size:", len(y_train))

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape)

Shape of the dataset: (393579, 2)
Training: (236147,) (236147,)
Validation: (78716,) (78716,)
Test: (78716,) (78716,)
Sentiment
Positive    184054
Negative     34241
Neutral      17852
Name: count, dtype: int64
Sentiment
Positive    60000
Negative    34241
Neutral     17852
Name: count, dtype: int64
New training size: 112093
(112093, 5000) (78716, 5000) (78716, 5000)


In [44]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}


y_train_encoded = y_train.map(label_map).to_numpy(dtype="int64")
y_val_encoded = y_val.map(label_map).to_numpy(dtype="int64")
y_test_encoded = y_test.map(label_map).to_numpy(dtype="int64")

print(y_train_encoded.shape)
print(y_val_encoded.shape)
print(y_test_encoded.shape)

(112093,)
(78716,)
(78716,)


In [45]:
class ReviewDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        row = self.features[index].toarray().flatten()
        return torch.tensor(row, dtype=torch.float32), self.labels[index]


train_dataset = ReviewDataset(X_train_tfidf, y_train_encoded)
val_dataset = ReviewDataset(X_val_tfidf, y_val_encoded)
test_dataset = ReviewDataset(X_test_tfidf, y_test_encoded)

#DataLoader for batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 1752
Validation batches: 1230
Testing batches: 1230


In [51]:
model=nn.Sequential(
    nn.Linear(5000, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 3)
)

print(model)

Sequential(
  (0): Linear(in_features=5000, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=3, bias=True)
)


In [ ]:
import numpy as np

class_counts = np.bincount(y_train_encoded)

class_weights = len(y_train_encoded) / (
    len(class_counts) * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

loss = nn.CrossEntropyLoss(weight=class_weights)

#or try using only loss = nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

itr=3

for epoch in range(itr):
    model.train()
    total_loss=0
    
    for X_batch, y_batch in train_loader:
        p=model(X_batch)
        l=loss(p,y_batch)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        
        total_loss+=l.item()
    
    avg_loss=total_loss/len(train_loader)
    print(f"Epoch {epoch + 1} | Average Loss: {avg_loss:.4f}")
        
        
            
        

Epoch 1 | Average Loss: 0.7323
Epoch 2 | Average Loss: 0.6290
Epoch 3 | Average Loss: 0.4599


In [53]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs=model(X_batch)
        predictions=torch.argmax(outputs,dim=1)
        
        # print("Predicted: ", predictions[:10].numpy())
        # print("Actual: ", y_batch[:10].numpy())
        
        all_predictions.extend(predictions.numpy())
        all_labels.extend(y_batch.numpy())
        
print(len(all_predictions))
print(len(all_labels))


78716
78716


In [54]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Negative", "Neutral", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))




Classification Report:
              precision    recall  f1-score   support

    Negative       0.59      0.72      0.65     11413
     Neutral       0.22      0.50      0.31      5951
    Positive       0.96      0.80      0.87     61352

    accuracy                           0.77     78716
   macro avg       0.59      0.67      0.61     78716
weighted avg       0.85      0.77      0.80     78716


Confusion Matrix:
[[ 8207  2238   968]
 [ 1655  2970  1326]
 [ 4091  8121 49140]]


In [ ]:
#testing

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.numpy())
        all_labels.extend(y_batch.numpy())
        

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Negative", "Neutral", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))


        
        


Classification Report:
              precision    recall  f1-score   support

    Negative       0.58      0.72      0.64     11413
     Neutral       0.22      0.48      0.30      5951
    Positive       0.96      0.80      0.87     61352

    accuracy                           0.77     78716
   macro avg       0.59      0.67      0.61     78716
weighted avg       0.85      0.77      0.80     78716


Confusion Matrix:
[[ 8184  2305   924]
 [ 1731  2884  1336]
 [ 4147  7955 49250]]


In [57]:
import os
import torch
import joblib

os.makedirs("../models", exist_ok=True)

torch.save(
    model.cpu().state_dict(),
    "../models/sentiment_mlp.pth"
)

joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

print("Model and TF-IDF vectorizer saved successfully!")

Model and TF-IDF vectorizer saved successfully!


In [58]:
print(os.path.exists("../models/sentiment_mlp.pth"))
print(os.path.exists("../models/tfidf_vectorizer.pkl"))

True
True
